# Movie Data Cleaning and EDA

Run `python -m src.run_pipeline` from the project root first. Configure `.env` and execute `sql/schema.sql` in MySQL Workbench.

In [ ]:
from src.config import get_engine
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

engine = get_engine()
movies = pd.read_sql('SELECT * FROM movies_cleaned', con=engine)
movies.head()

In [ ]:
print('Shape:', movies.shape)
print('\nMissing values:\n', movies.isna().sum())
movies.describe(include='all')

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(movies['rating'].dropna(), bins=20, kde=True)
plt.title('Distribution of Movie Ratings')
plt.tight_layout()
plt.show()

In [ ]:
top = movies[movies['vote_count'].fillna(0) >= 100].nlargest(10, 'rating')
display(top[['title', 'rating', 'vote_count']])
plt.figure(figsize=(10, 6))
sns.barplot(data=top, x='rating', y='title')
plt.title('Top Rated Movies (at least 100 votes)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=movies, x='popularity', y='rating')
plt.title('Popularity vs Rating')
plt.tight_layout()
plt.show()

In [ ]:
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
years = movies.dropna(subset=['release_date']).assign(year=lambda d: d['release_date'].dt.year).groupby('year').size()
years.tail(20).plot(kind='bar', figsize=(12, 5))
plt.title('Movie Records by Release Year')
plt.tight_layout()
plt.show()

## SQL checks

```sql
SELECT COUNT(*) FROM movies_cleaned;
SELECT language, COUNT(*) AS movie_count, ROUND(AVG(rating), 2) AS average_rating FROM movies_cleaned GROUP BY language ORDER BY movie_count DESC;
SELECT title, rating, vote_count FROM movies_cleaned WHERE vote_count >= 100 ORDER BY rating DESC LIMIT 10;
```